In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/acs_bexar_block_groups.csv', dtype={'GEOID': str})
print(f"Loaded {len(df)} block groups")
print(df[['GEOID','cost_burden_rate','severe_cost_burden_rate','overcrowding_rate']].head())

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/acs_bexar_block_groups.csv'

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('/Users/abilindemann/projects/sa-housing-food-insecurity/data/processed/acs_bexar_block_groups.csv', dtype={'GEOID': str})
print(f"Loaded {len(df)} block groups")
print(df[['GEOID','cost_burden_rate','severe_cost_burden_rate','overcrowding_rate']].head())

Loaded 1139 block groups
          GEOID  cost_burden_rate  severe_cost_burden_rate  overcrowding_rate
0  480291101001          0.513953                 0.177907           1.000000
1  480291101002          0.605547                 0.442219           0.606061
2  480291101003          0.561283                 0.199313           1.000000
3  480291103001          0.707368                 0.412632           0.750000
4  480291103002          0.220690                 0.000000           1.000000


In [5]:
# Check how many block groups have missing values in our three scoring columns
scoring_cols = ['cost_burden_rate', 'severe_cost_burden_rate', 'overcrowding_rate']
print("Null counts:")
print(df[scoring_cols].isnull().sum())
print(f"\nTotal block groups: {len(df)}")
print(f"Block groups with all three scores: {df[scoring_cols].dropna().shape[0]}")
print(f"\nScore ranges:")
print(df[scoring_cols].describe().round(3))

Null counts:
cost_burden_rate            41
severe_cost_burden_rate     41
overcrowding_rate          504
dtype: int64

Total block groups: 1139
Block groups with all three scores: 625

Score ranges:
       cost_burden_rate  severe_cost_burden_rate  overcrowding_rate
count          1098.000                 1098.000            635.000
mean              0.465                    0.227              0.275
std               0.265                    0.215              0.391
min               0.000                    0.000              0.000
25%               0.285                    0.000              0.000
50%               0.480                    0.199              0.000
75%               0.644                    0.348              0.536
max               1.000                    1.000              1.000


In [7]:
from sklearn.preprocessing import MinMaxScaler

# Normalize each component to 0-1 across all block groups
# We refit on non-null values so nulls don't distort the scale
def normalize(series):
    s = series.copy()
    valid = s.dropna()
    mn, mx = valid.min(), valid.max()
    if mx == mn:
        return s.apply(lambda x: 0 if pd.notna(x) else np.nan)
    return (s - mn) / (mx - mn)

df['cost_burden_norm']        = normalize(df['cost_burden_rate'])
df['severe_cost_burden_norm'] = normalize(df['severe_cost_burden_rate'])
df['overcrowding_norm']       = normalize(df['overcrowding_rate'])

# Composite score = mean of available components
score_cols = ['cost_burden_norm', 'severe_cost_burden_norm', 'overcrowding_norm']
df['housing_score'] = df[score_cols].mean(axis=1)  # skipna=True by default

# Track how many components went into each score
df['housing_score_components'] = df[score_cols].notna().sum(axis=1)

print("Housing score distribution:")
print(df['housing_score'].describe().round(3))
print(f"\nComponents used per block group:")
print(df['housing_score_components'].value_counts().sort_index())

Housing score distribution:
count    1108.000
mean        0.327
std         0.208
min         0.000
25%         0.186
50%         0.317
75%         0.464
max         1.000
Name: housing_score, dtype: float64

Components used per block group:
housing_score_components
0     31
1     10
2    473
3    625
Name: count, dtype: int64


In [9]:
# Classify block groups: top 25% = high housing insecurity
threshold = df['housing_score'].quantile(0.75)
df['housing_insecurity_high'] = (df['housing_score'] >= threshold).astype(int)

print(f"High housing insecurity threshold (75th percentile): {threshold:.3f}")
print(f"Block groups classified as high: {df['housing_insecurity_high'].sum()}")
print(f"Block groups classified as low:  {(df['housing_insecurity_high']==0).sum()}")

# Save output — keep GEOID, scores, raw rates, and key context columns
keep = [
    'GEOID',
    'cost_burden_rate', 'severe_cost_burden_rate', 'overcrowding_rate',
    'cost_burden_norm', 'severe_cost_burden_norm', 'overcrowding_norm',
    'housing_score', 'housing_score_components', 'housing_insecurity_high',
    'B19013_001E',  # median household income
    'pct_hispanic', 'poverty_rate',
    'B03002_003E', 'B03002_004E', 'B03002_001E',  # race/ethnicity counts
]
keep = [c for c in keep if c in df.columns]
out = df[keep].copy()

out.to_csv('/Users/abilindemann/projects/sa-housing-food-insecurity/data/processed/housing_scores.csv', index=False)
print(f"\nSaved housing_scores.csv with {len(out)} rows and {len(out.columns)} columns")

High housing insecurity threshold (75th percentile): 0.464
Block groups classified as high: 277
Block groups classified as low:  862

Saved housing_scores.csv with 1139 rows and 16 columns


## Notebook 01 — Housing Insecurity Score

**Output:** `data/processed/housing_scores.csv`

**Method:** Composite score (0–1) averaged across normalized components:
- Renter cost burden rate (% paying >30% income on housing)
- Severe cost burden rate (% paying >50%)
- Overcrowding rate (% of units with >1 person per room)

**Classification:** Top 25% (score ≥ 0.464) = high housing insecurity → 277 of 1,139 block groups

**Nulls:** 31 block groups had no scoreable data (likely unpopulated). 
504 block groups missing overcrowding (zero overcrowded units — scored on 2 components).